# YuE: Full-Song Music Generation on AWS Trainium (NxDI)

This notebook demonstrates how to run [YuE](https://github.com/multimodal-art-projection/YuE) (M-A-P/HKUST) on AWS Trainium using neuronx-distributed-inference (NxDI). YuE generates full songs from lyrics using a two-stage autoregressive pipeline:

- **Stage 1 (S1, 7B LLaMA)**: Generates coarse audio codec tokens from lyrics with Classifier-Free Guidance (CFG)
- **Stage 2 (S2, 1B LLaMA)**: Refines tokens via teacher-forcing (adds 7 codebook layers per frame)
- **xcodec_mini**: Decodes codec tokens to audio waveforms (CPU)

Both S1 and S2 are standard `LlamaForCausalLM` architectures with expanded vocabularies (~84K tokens), so NxDI's built-in `NeuronLlamaForCausalLM` works directly.

## Requirements

- **Instance**: trn2.3xlarge (or larger)
- **AMI**: Deep Learning AMI Neuron (Ubuntu 24.04) 20260227 or later (SDK 2.28)
- **Storage**: 300GB+ EBS (gp3)
- **Venv**: `/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/`

## What This Notebook Covers

1. **Setup**: Install dependencies, download models
2. **Compile**: Compile both S1 and S2 for Neuron with NKI kernel optimization
3. **Generate**: Run the full lyrics-to-audio pipeline
4. **Benchmark**: Compare performance across S2 batch sizes (1, 2, 4)

## Performance Summary (trn2.3xlarge)

| Configuration | Pipeline Time | RTF | vs L4 GPU |
|---|---|---|---|
| Baseline | 514s | 17.6x | 2.8x faster |
| + NKI kernels | ~440s | ~15x | 3.3x faster |
| + NKI + S2 bs=2 | ~467s | 16.0x | 3.1x faster |

## 1. Setup

Install dependencies and download models. Run this section once per instance.

In [ ]:
# Configuration -- change these as needed
MODEL_DIR = "/mnt/models"
VENV = "/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13"

In [ ]:
%%bash -s "$MODEL_DIR"
MODEL_DIR=$1

# Install extra Python packages
pip install -q soundfile omegaconf einops descript-audiotools 'protobuf>=3.20,<5'

# Create model directory
sudo mkdir -p "${MODEL_DIR}"
sudo chown ubuntu:ubuntu "${MODEL_DIR}"

In [ ]:
%%bash -s "$MODEL_DIR"
MODEL_DIR=$1
cd "${MODEL_DIR}"

echo "Downloading YuE S1 (7B) model..."
if [ ! -d "YuE-s1-7B-anneal-en-cot" ]; then
    huggingface-cli download m-a-p/YuE-s1-7B-anneal-en-cot \
        --local-dir YuE-s1-7B-anneal-en-cot \
        --local-dir-use-symlinks False
else
    echo "  Already exists, skipping."
fi

echo "Downloading YuE S2 (1B) model..."
if [ ! -d "YuE-s2-1B-general" ]; then
    huggingface-cli download m-a-p/YuE-s2-1B-general \
        --local-dir YuE-s2-1B-general \
        --local-dir-use-symlinks False
else
    echo "  Already exists, skipping."
fi

echo "Downloading xcodec_mini inference model..."
if [ ! -d "xcodec_mini_infer" ]; then
    huggingface-cli download m-a-p/xcodec_mini_infer \
        --local-dir xcodec_mini_infer \
        --local-dir-use-symlinks False
else
    echo "  Already exists, skipping."
fi

echo "Cloning YuE repository (inference code + tokenizer)..."
if [ ! -d "YuE" ]; then
    git clone https://github.com/multimodal-art-projection/YuE.git
else
    echo "  Already exists, skipping."
fi

echo "Setup complete. Models in: ${MODEL_DIR}"

In [ ]:
# Verify models are downloaded
import os

expected = ["YuE-s1-7B-anneal-en-cot", "YuE-s2-1B-general", "xcodec_mini_infer", "YuE"]
for d in expected:
    path = os.path.join(MODEL_DIR, d)
    exists = os.path.isdir(path)
    print(f"  {'OK' if exists else 'MISSING':>7}  {d}")
    assert exists, f"Missing: {path}"

print("\nAll models present.")

## 2. Copy Scripts to Model Directory

The pipeline uses subprocess isolation (S1 and S2 run in separate processes because NxDI models with different TP degrees cannot coexist). We copy the worker scripts alongside the models.

In [ ]:
import shutil

SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))  # notebook directory
# If running from the contrib/src directory, scripts are here.
# Otherwise, adjust this path to where the .py files are.

scripts = [
    "yue_e2e_neuron.py",
    "yue_stage1_worker.py",
    "yue_stage2_worker.py",
    "nki_mlp_patch.py",
]

for script in scripts:
    src = os.path.join(SCRIPT_DIR, script)
    if not os.path.exists(src):
        # Try src/ subdirectory (nxdi fork layout)
        src = os.path.join(SCRIPT_DIR, "src", script)
    dst = os.path.join(MODEL_DIR, script)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  Copied {script}")
    else:
        print(f"  SKIP {script} (not found at {src})")

print(f"\nScripts deployed to {MODEL_DIR}")

## 3. Define Inputs

Set genre tags and lyrics for the song. Lyrics use section tags like `[verse]`, `[chorus]`, `[bridge]`.

In [ ]:
GENRE = "pop electronic upbeat female vocal"

LYRICS = """[verse]
In the neon glow of a city night
Where the stars are hidden from our sight
We dance beneath the flickering lights
Chasing shadows, feeling so alive

[chorus]
Turn it up, let the music play
We're gonna dance the night away
Feel the rhythm in our hearts tonight
Everything's gonna be alright
"""

# Write input files
genre_path = os.path.join(MODEL_DIR, "genre.txt")
lyrics_path = os.path.join(MODEL_DIR, "lyrics.txt")

with open(genre_path, "w") as f:
    f.write(GENRE)
with open(lyrics_path, "w") as f:
    f.write(LYRICS)

print(f"Genre: {GENRE}")
print(f"Lyrics sections: {LYRICS.count('[')}") 
print(f"Written to: {genre_path}, {lyrics_path}")

## 4. Compile Models

This compiles both S1 (7B, TP=2) and S2 (1B, TP=1) to Neuron IR with NKI MLP TKG kernel optimization enabled. Compilation takes ~7 minutes total on first run.

We compile S2 at **multiple batch sizes** (1, 2, 4) so we can benchmark them later.

### NKI Kernel Optimization

The `--nki-kernels` flag enables NxDI's fused NKI MLP kernel for token generation, which keeps all MLP intermediates in on-chip SRAM (SBUF), eliminating 5-6 HBM round-trips per layer. This gives a **20% speedup** on token generation.

In [ ]:
%%bash -s "$MODEL_DIR"
MODEL_DIR=$1
cd "${MODEL_DIR}"

echo "=== Compiling S1 + S2 (NKI enabled, S2 batch_size=1) ==="
echo "This takes ~7 minutes on first run."
echo ""

python yue_e2e_neuron.py \
    --genre_txt genre.txt \
    --lyrics_txt lyrics.txt \
    --seed 123 \
    --nki-kernels \
    --s2-batch-size 1 \
    --run_n_segments 2 \
    --output_dir ./output_bs1

echo ""
echo "=== Compilation + generation complete (bs=1) ==="

## 5. Generate with S2 Batch Size = 2 (Recommended)

Batch size 2 processes two S2 chunks simultaneously, giving the best end-to-end pipeline time. This requires a separate S2 compilation.

In [ ]:
%%bash -s "$MODEL_DIR"
MODEL_DIR=$1
cd "${MODEL_DIR}"

echo "=== Running with S2 batch_size=2 (NKI enabled) ==="
echo "S1 uses cached compilation. S2 bs=2 compiles on first run (~4 min)."
echo ""

python yue_e2e_neuron.py \
    --genre_txt genre.txt \
    --lyrics_txt lyrics.txt \
    --seed 123 \
    --nki-kernels \
    --s2-batch-size 2 \
    --skip-compile \
    --run_n_segments 2 \
    --output_dir ./output_bs2

echo ""
echo "=== Generation complete (bs=2) ==="

## 6. Generate with S2 Batch Size = 4

Higher batch sizes increase per-core throughput but model load overhead grows. Batch size 4 typically has higher total pipeline time than batch size 2 due to this tradeoff.

In [ ]:
%%bash -s "$MODEL_DIR"
MODEL_DIR=$1
cd "${MODEL_DIR}"

echo "=== Running with S2 batch_size=4 (NKI enabled) ==="
echo "S1 uses cached compilation. S2 bs=4 compiles on first run (~4 min)."
echo ""

python yue_e2e_neuron.py \
    --genre_txt genre.txt \
    --lyrics_txt lyrics.txt \
    --seed 123 \
    --nki-kernels \
    --s2-batch-size 4 \
    --skip-compile \
    --run_n_segments 2 \
    --output_dir ./output_bs4

echo ""
echo "=== Generation complete (bs=4) ==="

## 7. Benchmark Results

Parse the pipeline summaries from each run and compare performance across batch sizes.

In [ ]:
import re
import subprocess
import sys

def run_benchmark(model_dir, batch_size, output_dir):
    """Run the pipeline and capture the summary."""
    cmd = [
        sys.executable, os.path.join(model_dir, "yue_e2e_neuron.py"),
        "--genre_txt", os.path.join(model_dir, "genre.txt"),
        "--lyrics_txt", os.path.join(model_dir, "lyrics.txt"),
        "--seed", "123",
        "--nki-kernels",
        "--s2-batch-size", str(batch_size),
        "--skip-compile",
        "--run_n_segments", "2",
        "--output_dir", output_dir,
    ]
    env = os.environ.copy()
    env["MODEL_DIR"] = model_dir
    
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=model_dir, env=env)
    output = result.stdout + result.stderr
    
    # Parse key metrics from output
    metrics = {"batch_size": batch_size}
    
    m = re.search(r"Stage 1 \(S1 7B\): ([\d.]+)s", output)
    if m: metrics["s1_time"] = float(m.group(1))
    
    m = re.search(r"Stage 2 \(S2 1B\): ([\d.]+)s", output)
    if m: metrics["s2_time"] = float(m.group(1))
    
    m = re.search(r"Stage 3 \(decode\): ([\d.]+)s", output)
    if m: metrics["decode_time"] = float(m.group(1))
    
    m = re.search(r"Total pipeline: ([\d.]+)s", output)
    if m: metrics["total_time"] = float(m.group(1))
    
    m = re.search(r"Audio duration: ([\d.]+)s", output)
    if m: metrics["audio_duration"] = float(m.group(1))
    
    m = re.search(r"Real-time factor: ([\d.]+)x", output)
    if m: metrics["rtf"] = float(m.group(1))
    
    if result.returncode != 0:
        print(f"WARNING: bs={batch_size} returned non-zero exit code {result.returncode}")
        print(output[-500:])
    
    return metrics, output

print("Benchmark function defined. Run the next cell to collect results.")

In [ ]:
# Run benchmarks for each batch size
# NOTE: Each run takes ~7-10 minutes. The S1 compilation is cached after the first run.
#       S2 needs separate compilation per batch size (cached after first run of each).

batch_sizes = [1, 2, 4]
results = []

for bs in batch_sizes:
    print(f"\n{'='*60}")
    print(f"Running benchmark: S2 batch_size={bs}")
    print(f"{'='*60}")
    
    output_dir = os.path.join(MODEL_DIR, f"output_bench_bs{bs}")
    metrics, output = run_benchmark(MODEL_DIR, bs, output_dir)
    results.append(metrics)
    
    if "total_time" in metrics:
        print(f"  bs={bs}: total={metrics['total_time']:.1f}s, "
              f"S1={metrics.get('s1_time', '?')}s, "
              f"S2={metrics.get('s2_time', '?')}s, "
              f"RTF={metrics.get('rtf', '?')}x")
    else:
        print(f"  bs={bs}: Failed to parse metrics")
        print(output[-300:])

print(f"\nAll benchmarks complete.")

In [ ]:
# Display results as a table
print(f"{'='*75}")
print(f"BENCHMARK RESULTS (NKI kernels enabled, seed=123, 2 segments)")
print(f"{'='*75}")
print(f"{'S2 Batch Size':>13} | {'S1 Time':>8} | {'S2 Time':>8} | {'Decode':>7} | {'Total':>8} | {'RTF':>6} | {'Audio':>6}")
print(f"{'-'*13}-+-{'-'*8}-+-{'-'*8}-+-{'-'*7}-+-{'-'*8}-+-{'-'*6}-+-{'-'*6}")

for r in results:
    bs = r.get('batch_size', '?')
    s1 = f"{r['s1_time']:.1f}s" if 's1_time' in r else '?'
    s2 = f"{r['s2_time']:.1f}s" if 's2_time' in r else '?'
    dec = f"{r['decode_time']:.1f}s" if 'decode_time' in r else '?'
    tot = f"{r['total_time']:.1f}s" if 'total_time' in r else '?'
    rtf = f"{r['rtf']:.1f}x" if 'rtf' in r else '?'
    dur = f"{r['audio_duration']:.1f}s" if 'audio_duration' in r else '?'
    print(f"{bs:>13} | {s1:>8} | {s2:>8} | {dec:>7} | {tot:>8} | {rtf:>6} | {dur:>6}")

print(f"{'='*75}")
print(f"\nKey insights:")
print(f"  - S1 time is constant across batch sizes (CFG uses fixed bs=2)")
print(f"  - S2 compute improves with batch size, but model load overhead grows")
print(f"  - Batch size 2 typically gives the best end-to-end pipeline time")

## 8. Listen to Generated Audio

Play the generated audio files. The pipeline produces separate vocal and instrumental tracks, plus a mixed version.

In [ ]:
import glob
from IPython.display import Audio, display, HTML

# Find the most recent output
for bs in [2, 1, 4]:  # prefer bs=2 output
    audio_dir = os.path.join(MODEL_DIR, f"output_bs{bs}", "audio")
    if os.path.isdir(audio_dir):
        wavs = glob.glob(os.path.join(audio_dir, "*.wav"))
        if wavs:
            break

if not wavs:
    # Try benchmark outputs
    for bs in [2, 1, 4]:
        audio_dir = os.path.join(MODEL_DIR, f"output_bench_bs{bs}", "audio")
        if os.path.isdir(audio_dir):
            wavs = glob.glob(os.path.join(audio_dir, "*.wav"))
            if wavs:
                break

if wavs:
    print(f"Audio files from: {audio_dir}\n")
    for wav_path in sorted(wavs):
        name = os.path.basename(wav_path)
        label = name.split("_")[0].capitalize()  # vocals, instrumentals, or mix
        display(HTML(f"<h4>{label}</h4>"))
        display(Audio(wav_path))
else:
    print("No audio files found. Run the generation cells first.")

## 9. Architecture Deep Dive

### Pipeline Architecture

```
                    +-----------------+
  genre.txt ------->|                 |
  lyrics.txt ------>|  Orchestrator   |----> vocals.wav
                    | (yue_e2e_neuron)|----> instrumentals.wav
                    |                 |----> mix.wav
                    +--------+--------+
                             |
              +--------------+--------------+
              |              |              |
     +--------v-------+ +---v----+ +-------v--------+
     | S1 Worker (7B) | | S2 (1B)| | xcodec_mini    |
     | TP=2, NxDI     | | TP=1   | | CPU decode     |
     | CFG bs=2       | | NxDI   | |                |
     +----------------+ +--------+ +----------------+
       subprocess         subprocess   main process
```

### Why Subprocesses?

NxDI models with different TP degrees cannot coexist in the same process -- the Neuron runtime segfaults during warmup. S1 (TP=2) and S2 (TP=1) each run in their own subprocess, with arguments passed via the `YUE_STAGE_ARGS` environment variable.

### NKI MLP Kernel Routing

The `nki_mlp_patch.py` monkeypatch routes MLP computation based on batch * sequence length:

- **Token Generation** (B*S <= 128): Uses NxDI's `nki_mlp_tkg_isa_kernel` which fuses RMSNorm + Gate/Up + SiLU + Down into a single kernel in SBUF. No dimension limits.
- **Context Encoding** (B*S > 128, I <= 4096): Uses the built-in `mlp_isa_kernel`.
- **Context Encoding** (B*S > 128, I > 4096): Manual matmul fallback, bypassing the compiler's 4096 intermediate dimension limit.

### Classifier-Free Guidance (CFG)

CFG is critical for vocal quality. The implementation uses a custom generation loop (not HF `generate()`) because:
1. CFG requires blending logits from conditional (row 0) and unconditional (row 1) before sampling
2. The **same** sampled token must be fed to **both** rows to keep KV caches synchronized
3. HF `generate()` with batch_size=2 samples independently per row, causing divergence

## 10. Customization Options

The pipeline supports many options via command-line arguments:

```
python yue_e2e_neuron.py --genre_txt GENRE --lyrics_txt LYRICS [OPTIONS]

Required:
  --genre_txt FILE          Genre description file
  --lyrics_txt FILE         Lyrics file with [verse]/[chorus] sections

Optional:
  --output_dir DIR          Output directory (default: ./output_neuron)
  --seed INT                Random seed (default: 42)
  --max_new_tokens INT      Max tokens per segment (default: 3000)
  --run_n_segments INT      Number of lyrics segments (default: 2)
  --skip-compile            Skip Neuron compilation (use cached)
  --nki-kernels             Enable NKI MLP TKG fused kernels (20% speedup)
  --s2-batch-size N         S2 batch size (default: 1, try 2 for best pipeline time)
  --s1-tp-degree N          S1 TP degree (default: 2)
  --no-cfg                  Disable CFG (faster, lower vocal quality)
  --no-kv-cache             Disable KV-cache S2 optimization (use legacy loop)
  --guidance-scale-first F  CFG scale, first segment (default: 1.5)
  --guidance-scale-rest F   CFG scale, subsequent segments (default: 1.2)
  --rescale                 Rescale audio to max amplitude
```

### Try Different Genres

Edit the genre and lyrics cells above and re-run. Some genre examples:

- `pop electronic upbeat female vocal`
- `rock guitar male vocal energetic drums`
- `jazz piano smooth saxophone nightclub`
- `hip hop beat bass heavy urban`

## 11. Cleanup

Remove compiled models and output files to free disk space.

In [ ]:
# Uncomment to clean up compiled models (~5GB)
# !rm -rf {MODEL_DIR}/compiled/

# Uncomment to clean up all output audio
# !rm -rf {MODEL_DIR}/output_*

print("Cleanup commands are commented out. Uncomment and run to free disk space.")